In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


In [ ]:
DATA = Path("/user/ab5405/summeraliaclimate/code/energy_uncertainty/data")
BENCH = Path("/user/ab5405/summeraliaclimate/code/energy_uncertainty/output_original")

# benchmark panel (original)
gmfd_old = pd.read_stata("/user/ab5405/summeraliaclimate/code/energy_consumption/energy_data_release_2021oct21/DATA/regression/GMFD_TINV_clim_regsort.dta")

# your rebuilt climate
gmfd_new = pd.read_csv(DATA / "country_climate" / "GMFD" / "GMFD_country_year_1971_2010.csv")

# harmonize
gmfd_old = gmfd_old.rename(columns={"country":"iso"}) if "country" in gmfd_old else gmfd_old

cols = ["iso","year","precip1_GMFD","precip2_GMFD","hdd20_GMFD","cdd20_GMFD"]
merged = (
    gmfd_old[cols]
    .merge(gmfd_new[cols], on=["iso","year"], suffixes=("_old","_new"))
)


In [ ]:
def compare(var):
    out = {}
    o, n = f"{var}_old", f"{var}_new"
    out["corr"] = merged[o].corr(merged[n])
    out["mean_old"] = merged[o].mean()
    out["mean_new"] = merged[n].mean()
    out["reldiff"] = ((merged[n] - merged[o]) / merged[o]).mean()
    return out

for v in ["precip1_GMFD","precip2_GMFD","hdd20_GMFD","cdd20_GMFD"]:
    print(v, compare(v))


In [ ]:
merged = merged.sort_values(["iso","year"])
for v in ["precip1_GMFD","precip2_GMFD"]:
    merged[f"FD_{v}_old"] = merged.groupby("iso")[f"{v}_old"].diff()
    merged[f"FD_{v}_new"] = merged.groupby("iso")[f"{v}_new"].diff()

merged[[c for c in merged.columns if "FD_precip" in c]].dropna().head(10)


In [ ]:
OLD = Path("/user/ab5405/summeraliaclimate/code/energy_uncertainty/output_original/sters")
NEW = Path("/user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters")

old = pd.read_csv(OLD / "FD_FGLS_inter_TINV_clim_quadinter_coeff.csv")
new = pd.read_csv(NEW / "FD_FGLS_inter_TINV_clim_quadinter_GMFD_coeff.csv")

comp = old.merge(new, on="parm", how="inner", suffixes=("_old","_new"))
comp["abs_diff"] = comp.beta_new - comp.beta_old
comp["rel_diff"] = comp.abs_diff / comp.beta_old


In [ ]:
precip = comp[comp.parm.str.contains("FD_precip")]
precip.sort_values("abs_diff")
